<a href="https://colab.research.google.com/github/resthiana12/Kelainan_Psikologis_CNN/blob/main/SDKP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**#Install Package yang Dibutuhkan**

In [ ]:
pip install sastrawi

In [ ]:
pip install swifter

**#Load Library**

In [ ]:
#load library
import nltk                             
import matplotlib.pyplot as plt           
import random  
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('words')
import numpy as np
import pandas as pd
import re                                  
import string                             
from nltk.corpus import stopwords 
from nltk.stem import PorterStemmer
from nltk.tokenize import TweetTokenizer 
from nltk import pos_tag, word_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import words
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import swifter
from wordcloud import WordCloud

**Preprocessing**

In [ ]:
def ubah_text_lower(text):
    return text.lower()
    
stop = pd.read_csv("drive/MyDrive/stopwords.txt", names= ["stopwords"], header = None)
stop_words_corpus = list(stopwords.words('indonesian'))
stop_from_list = stop['stopwords'].to_list()
def hapus_stopwords(post):
  filtered_words = [w for w in post if w not in stop_words_corpus]
  filtered_words = [w for w in filtered_words if w not in stop_from_list]
  return filtered_words

def preprocess_filtered(post):
    #Remove 'b
    posted = re.sub(r"b'", ' ', post)
    # Remove urls dan trash
    posted = re.sub(r"http\S+|\\x\S+|www\S+|https\S+|dot|com", ' ', posted, flags=re.MULTILINE)
    # Remove user @ references and '#' from post
    posted = re.sub(r'\@\w+|\#|\d+', ' ', post)
    return posted

def tokenkan(text):
    return word_tokenize(text)

normalizad_word = pd.read_excel("drive/MyDrive/normalisasi.xlsx")
normalizad_word_dict = {}
for index, row in normalizad_word.iterrows():
    if row[0] not in normalizad_word_dict:
        normalizad_word_dict[row[0]] = row[1] 

def normalized_term(document):
    return [normalizad_word_dict[term] if term in normalizad_word_dict else term for term in document]

def hapus_punct_2(text):
  text_nopunct = ''
  text_nopunct = re.sub('['+string.punctuation+']', '', text)
  return text_nopunct

def hapus_punct(text):
    PUNCT_TO_REMOVE = string.punctuation
    punct = text.translate(str.maketrans('', '', PUNCT_TO_REMOVE))
    return punct

#Import Libraries

In [ ]:
from gensim.models.word2vec import Word2Vec
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from keras.layers import Dense, Dropout, Conv1D, MaxPool1D, GlobalMaxPool1D, Embedding, Activation
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from sklearn import preprocessing

In [ ]:
df = pd.read_csv('drive/MyDrive/data_psyco/Text_Preprocessing_Depres_Suicide_990_Terbaru_Fix_3.csv').astype(str)
#df = pd.read_csv('drive/MyDrive/data_psyco/Text_Preprocessing_Depres_Suicide_Fix.csv').astype(str)
#df.drop(['Unnamed: 0', 'Postingan', 'Postingan_Tokenkan', 'Postingan_Normalized'], axis=1, inplace=True)
#df.drop(['Unnamed: 0', 'Postingan'], axis=1, inplace=True)
df.columns = ['Label', 'Post']
#df['Message'] = df['Message'].apply(remove_punct)
df.head()

In [ ]:
#check the distribution of lebels
labels = df['Label']
labels_count = labels.value_counts()
labels_count.plot(kind="bar")
print(labels.value_counts())

In [ ]:
post = []
for i in df['Post']:
    post.append(i.split())
print(post[:2])

In [ ]:
  word2vec_model = Word2Vec(post, size=500, window=3, min_count=1, workers=15)
print(word2vec_model)

In [ ]:
token = Tokenizer(2772)
token.fit_on_texts(df['Post'])
text = token.texts_to_sequences(df['Post'])
text = pad_sequences(text, 100)
print(text[:2])

In [ ]:
le = preprocessing.LabelEncoder()
y = le.fit_transform(df['Label'])
y = to_categorical(y)
y[:2]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(np.array(text), y, test_size=0.2, stratify=y)

In [ ]:
print(word2vec_model.wv.get_keras_embedding(True))

In [ ]:
keras_model = Sequential()
keras_model.add(word2vec_model.wv.get_keras_embedding(True))
keras_model.add(Dropout(0.2))
keras_model.add(Conv1D(filters=50, kernel_size=3, activation='relu', padding='same', strides=1))
keras_model.add(Conv1D(filters=50, kernel_size=3, activation='relu', padding='same', strides=1))
keras_model.add(MaxPool1D())
keras_model.add(Dropout(0.2))
keras_model.add(Conv1D(filters=100, kernel_size=3, activation='relu', padding='same', strides=1))
keras_model.add(Conv1D(filters=100, kernel_size=3, activation='relu', padding='same', strides=1))
keras_model.add(MaxPool1D())
keras_model.add(Dropout(0.2))
keras_model.add(Conv1D(filters=200, kernel_size=3, activation='relu', padding='same', strides=1))
keras_model.add(Conv1D(filters=200, kernel_size=3, activation='relu', padding='same', strides=1))
keras_model.add(GlobalMaxPool1D())
keras_model.add(Dropout(0.2))
keras_model.add(Dense(200))
keras_model.add(Activation('relu'))
keras_model.add(Dropout(0.2))
keras_model.add(Dense(2))
keras_model.add(Activation('sigmoid'))
keras_model.compile(loss='binary_crossentropy', metrics=['acc'], optimizer='adam')
keras_model.summary()

In [ ]:
history = keras_model.fit(x_train, y_train, batch_size=16, epochs=4, validation_data=(x_test, y_test))

In [ ]:
#analyze the results
score, acc = keras_model.evaluate(x_test, y_test, verbose = 2, batch_size=16)
y_pred = keras_model.predict(x_test)
#print(y_pred)

In [ ]:
#Other accuracy metrices
#y_pred = (y_pred > 0.5)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

cm=confusion_matrix(y_test.argmax(axis=1),y_pred.argmax(axis=1))
ax=sns.heatmap(cm,annot=True,cmap='Blues',fmt='')
ax.set_title('Confusion Matrix')
ax.set_xlabel('y_test')
ax.set_ylabel('y_pred')

#F1 Score, Recall and Precision
print(classification_report(y_test.argmax(axis=1), y_pred.argmax(axis=1)))

In [ ]:
plt.style.use('ggplot')

def plot_history(history):
    accuracy = history.history['acc']
    val_accuracy = history.history['val_acc']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    x = range(1, len(accuracy) + 1)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(x, accuracy, 'b', label='Training accuracy')
    plt.plot(x, val_accuracy, 'r', label='Validation accuracy')
    plt.title('Training and validation accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(x, loss, 'b', label='Training loss')
    plt.plot(x, val_loss, 'r', label='Validation loss')
    plt.title('Training and validation loss')
    plt.legend()

%matplotlib inline
plot_history(history)

In [ ]:
keras_model.save('drive/MyDrive/data_psyco/otw_sidang_sigmoid/sdkp.h5')

# Import Libraries

In [ ]:
def text_preprocessing(text):
  df['text_lower'] = text.apply(ubah_text_lower)
  df['filtered'] = df['text_lower'].apply(preprocess_filtered)
  df['token'] = df['filtered'].apply(tokenkan)
  df['normalized'] = df['token'].apply(normalized_term)
  df['normalized'] = df['normalized'].apply(hapus_stopwords)

  # create stemmer
  factory = StemmerFactory()
  stemmer = factory.create_stemmer()

  # stemmed
  def stemmed_wrapper(term):
      return stemmer.stem(term)

  term_dict = {}

  for document in df['normalized']:
      for term in document:
          if term not in term_dict:
              term_dict[term] = ' '
              
  #print(len(term_dict))
  print("------------------------")

  for term in term_dict:
      term_dict[term] = stemmed_wrapper(term)
      print(term,":" ,term_dict[term])
      
  #print(term_dict)
  print("------------------------")


  # apply stemmed term to dataframe
  def stemmed_term(document):
      return [term_dict[term] for term in document]

  df['stemmed'] = df['normalized'].swifter.apply(stemmed_term)
  #print(df['stemmed'])

  df['stemmed'] = df['stemmed'].astype(str)
  df['stemmed'] = df['stemmed'].apply(hapus_punct_2)

In [ ]:
Xo = 'Aku ingin ice cream'
Yo = 'lelah dengan hidup ini. Nak mati aku dah penat #depresi'
datax = {'Postingan':[Yo]}
df = pd.DataFrame(datax)
text_preprocessing(df['Postingan'])
df.head()

In [ ]:
X_sample = token.texts_to_sequences(df['stemmed'])
print(X_sample)
X_sample = pad_sequences(X_sample, 100)
print(X_sample)
y_sample = keras_model.predict(X_sample)
print(y_sample)
hasil = np.argmax(y_sample)

print(hasil)
if (hasil == 0) :
  print('tidak mengidap gangguan psikologis')
else:
  print('mengidap gangguan psikologis')